# M1 Baseline — Moment-DETR Training on QVHighlights Subset

**Stage:** 3 of 4 in the thumbnail consistency pipeline  
**Goal:** Train Moment-DETR (CLIP-only config) on the 2000-sample stratified subset and establish baseline R1@0.5, R1@0.7, and mAP numbers.

**What this notebook does:**
1. Extracts the official CLIP feature files
2. Audits that every video in the subset has a matching feature file
3. Builds a clean subset JSONL for the dataloader
4. Runs training via the original `train.py` script (treated as a library/tool)
5. Parses and displays the final evaluation metrics
6. Runs a saliency sanity check — visualises per-clip scores on 5 val examples

**What it does NOT do:**  
Re-implement Moment-DETR. The model code is used as-is from the original repo.  
The research contribution of this project is the pipeline that wraps it.


In [ ]:
# Install dependencies
# Run once per Colab session; skip if already installed.
!pip install -q einops
!pip install -q ftfy regex tqdm
!pip install -q git+https://github.com/openai/CLIP.git

import subprocess, sys
def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
# ── Project paths on Drive ───────────────────────────────────────────────────
PROJECT_ROOT   = Path("/content/drive/MyDrive/thumbnail_project")
REPO_DIR       = PROJECT_ROOT / "moment_detr"           # git repo root (has train.py)
FEATURES_TAR   = PROJECT_ROOT / "moment_detr_features.tar.gz"
TEXT_FEAT_ZIP  = PROJECT_ROOT / "clip_aug_text_features.zip"
PROCESSED_DIR  = PROJECT_ROOT / "processed"
# ── Where features will live after extraction ────────────────────────────────
# Extracted inside REPO_DIR so relative paths in train.py work by default.
FEAT_ROOT       = REPO_DIR / "features"
CLIP_FEAT_DIR   = FEAT_ROOT / "clip_features"
TEXT_FEAT_DIR   = FEAT_ROOT / "clip_text_features"

# ── Annotation files (already in repo) ──────────────────────────────────────
DATA_DIR        = REPO_DIR / "data"
FULL_TRAIN_JSONL = DATA_DIR / "highlight_train_release.jsonl"
VAL_JSONL        = DATA_DIR / "highlight_val_release.jsonl"
SUBSET_JSONL     = PROCESSED_DIR / "train_processed.jsonl"   # your 2000-sample subset

# ── Results ──────────────────────────────────────────────────────────────────
RESULTS_ROOT    = PROJECT_ROOT / "m1_results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print("REPO_DIR       :", REPO_DIR)
print("FEATURES_TAR   :", FEATURES_TAR.exists())
print("TEXT_FEAT_ZIP  :", TEXT_FEAT_ZIP.exists())
print("SUBSET_JSONL   :", SUBSET_JSONL.exists())

## Step 1 — Extract Feature Files

Run once. Skips extraction if already done.

`moment_detr_features.tar.gz` contains CLIP and SlowFast visual features (one `.npz` per video).  
`clip_aug_text_features.zip` contains per-query CLIP text embeddings.

In [ ]:
import os, tarfile, zipfile
# ── Extract video features ────────────────────────────────────────────────────
if not CLIP_FEAT_DIR.exists():
    print(f"Extracting {FEATURES_TAR.name} → {FEAT_ROOT} ...")
    with tarfile.open(FEATURES_TAR, "r:gz") as tar:
        tar.extractall(REPO_DIR)          # extracts to features/ inside REPO_DIR
    print("  Done.")
else:
    n = len(list(CLIP_FEAT_DIR.glob("*.npz")))
    print(f"CLIP features already extracted: {n} files in {CLIP_FEAT_DIR}")

# ── Extract text features ─────────────────────────────────────────────────────
if not TEXT_FEAT_DIR.exists():
    print(f"Extracting {TEXT_FEAT_ZIP.name} → {FEAT_ROOT} ...")
    with zipfile.ZipFile(TEXT_FEAT_ZIP, "r") as z:
        z.extractall(FEAT_ROOT)
    print("  Done.")
else:
    n = len(list(TEXT_FEAT_DIR.glob("*.npz")))
    print(f"Text features already extracted: {n} files in {TEXT_FEAT_DIR}")

# ── Quick shape inspection ────────────────────────────────────────────────────
import numpy as np
sample_vid = next(CLIP_FEAT_DIR.glob("*.npz"), None)
sample_txt = next(TEXT_FEAT_DIR.glob("*.npz"), None)
if sample_vid:
    f = np.load(sample_vid)
    print(f"\nVideo feature sample  : {sample_vid.name}")
    print(f"  keys : {f.files}   shape : {f[f.files[0]].shape}")

if sample_txt:
    f = np.load(sample_txt)
    print(f"\nText feature sample   : {sample_txt.name}")
    for k in f.files:
        print(f"  key='{k}'  shape={f[k].shape}")

## Step 2 — Audit Subset Coverage

Every video in your 2000-sample subset must have a matching `.npz` feature file.  
Missing features cause silent `FileNotFoundError` during training.

In [ ]:
import json
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

# Load your subset
subset = load_jsonl(SUBSET_JSONL)
print(f"Subset records loaded : {len(subset)}")

# Check which vids have feature files
# Feature filename format: {vid}.npz  where vid = youtube_id_start_end
missing_clip  = []
missing_text  = []
present_both  = []

for r in subset:
    vid  = r["vid"]
    qid  = r["qid"]
    clip_path = CLIP_FEAT_DIR / f"{vid}.npz"
    text_path = TEXT_FEAT_DIR / f"qid{qid}.npz"

    has_clip = clip_path.exists()
    has_text = text_path.exists()

    if not has_clip:
        missing_clip.append(vid)
    if not has_text:
        missing_text.append(qid)
    if has_clip and has_text:
        present_both.append(r)

print(f"\nRecords with BOTH features  : {len(present_both)} / {len(subset)}")
print(f"Missing CLIP video features : {len(missing_clip)}")
print(f"Missing text features       : {len(missing_text)}")

if missing_clip[:5]:
    print(f"\nFirst 5 missing video vids  : {missing_clip[:5]}")
if missing_text[:5]:
    print(f"First 5 missing text qids   : {missing_text[:5]}")


## Step 3 — Build Clean Subset JSONL

Filter the subset to only records that have both feature files. Save as `m1_train_subset.jsonl` in the format `train.py` expects.

The format is identical to `highlight_train_release.jsonl` — your `train_processed.jsonl` already follows this because it was derived from the official annotations.

In [ ]:
M1_TRAIN_JSONL = PROCESSED_DIR / "m1_train_subset.jsonl"
# The fields that train.py reads: qid, vid, query, relevant_windows,
# saliency_scores, relevant_clip_ids, duration
# train_processed.jsonl has all of these (it's a superset — extra fields like
# frame_path and clip_score are silently ignored by the dataloader)
if not M1_TRAIN_JSONL.exists():
    present_qids = {r["qid"] for r in present_both}
    clean = [r for r in subset if r["qid"] in present_qids]

    with open(M1_TRAIN_JSONL, "w") as f:
        for r in clean:
            f.write(json.dumps(r) + "\n")

    print(f"Saved {len(clean)} records → {M1_TRAIN_JSONL.name}")
else:
    clean = load_jsonl(M1_TRAIN_JSONL)
    print(f"Already exists: {len(clean)} records in {M1_TRAIN_JSONL.name}")

# Verify a record has all required keys
required = {"qid", "vid", "query", "relevant_windows",
            "saliency_scores", "relevant_clip_ids"}
bad = [r for r in clean if not required.issubset(r.keys())]
print(f"Records missing required fields: {len(bad)}  (expected 0)")


## Step 4 — Run M1 Baseline Training

Calls the original `train.py` via subprocess with the CLIP-only configuration.  
Expected wall time on a Colab T4: **3–5 hours** for 200 epochs on 2000 samples.

**Flags explained:**
- `--ctx_mode video_tef` — append temporal endpoint features (2 dims) to each clip feature
- `--contrastive_align_loss` — enabled by a flag, already implemented, improves ~1–2% R1@0.5
- `--data_ratio 1.0` — use 100% of your custom JSONL (which IS the subset)
- `--exp_id` — names the results subdirectory

In [ ]:
import subprocess, sys, os

EXP_ID = "clip_tef_subset2000_contrastive"   # change to distinguish runs

cmd = [
    sys.executable,
    str(REPO_DIR / "moment_detr" / "train.py"),
    "--dset_name",           "hl",
    "--ctx_mode",            "video_tef",
    "--train_path",          str(M1_TRAIN_JSONL),
    "--eval_path",           str(VAL_JSONL),
    "--eval_split_name",     "val",
    "--v_feat_dirs",         str(CLIP_FEAT_DIR),
    "--v_feat_dim",          "512",
    "--t_feat_dir",          str(TEXT_FEAT_DIR) + "/",
    "--t_feat_dim",          "512",
    "--bsz",                 "32",
    "--n_epoch",             "200",
    "--lr",                  "1e-4",
    "--wd",                  "1e-4",
    "--grad_clip",           "0.1",
    "--max_v_l",             "75",
    "--max_q_l",             "32",
    "--num_queries",         "10",
    "--contrastive_align_loss",             # Tier 1 improvement: no engineering cost
    "--contrastive_align_loss_coef", "0.02",
    "--results_root",        str(RESULTS_ROOT),
    "--exp_id",              EXP_ID,
]

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_DIR) + ":" + env.get("PYTHONPATH", "")

print("Command:")
print(" ".join(cmd))
print(f"\nResults will be saved to: {RESULTS_ROOT / EXP_ID}")
print("\nStarting training...\n")

result = subprocess.run(cmd, env=env, cwd=str(REPO_DIR))
print(f"\nTraining finished with exit code: {result.returncode}")


## Step 5 — Parse and Display Results

Reads the eval log written by `train.py` and surfaces the best epoch metrics.

In [ ]:
import re

EXP_DIR  = RESULTS_ROOT / EXP_ID
LOG_PATH = EXP_DIR / "train.log.txt"

if not LOG_PATH.exists():
    # train.py may save under a date-stamped subdirectory
    candidates = sorted(RESULTS_ROOT.rglob("train.log.txt"))
    if candidates:
        LOG_PATH = candidates[-1]
        EXP_DIR  = LOG_PATH.parent
        print(f"Found log at: {LOG_PATH}")
    else:
        raise FileNotFoundError("No train.log.txt found. Did training complete?")

with open(LOG_PATH) as f:
    log_text = f.read()

# Extract best-epoch metrics using regex
# Format: [best epoch] MR/HL -> R1@0.5 xx.xx ...
best_block_pattern = re.compile(
    r"INFO.*?Best.*?R1@0\.5=([0-9.]+).*?R1@0\.7=([0-9.]+).*?mAP@0\.5=([0-9.]+)",
    re.DOTALL
)
matches = best_block_pattern.findall(log_text)

# Fallback: find the last eval metrics block
eval_pattern = re.compile(
    r"R1@0\.5=([0-9.]+).*?R1@0\.7=([0-9.]+).*?mAP@0\.5=([0-9.]+)",
    re.DOTALL
)
all_evals = eval_pattern.findall(log_text)

print("=" * 55)
print("  M1 BASELINE RESULTS — CLIP+TEF, 2000-sample subset")
print("=" * 55)

if all_evals:
    best = max(all_evals, key=lambda x: float(x[0]))
    print(f"  Best R1@0.5 : {best[0]}")
    print(f"  Best R1@0.7 : {best[1]}")
    print(f"  Best mAP    : {best[2]}")
else:
    print("  Could not parse metrics. Check log manually:")
    print(LOG_PATH)

print("=" * 55)
print(f"\nReference (official full dataset, CLIP+TEF):")
print(f"  R1@0.5 = 53.23 | R1@0.7 = 34.97 | mAP = 30.58")
print(f"\nExpected for 2000-sample subset:")
print(f"  R1@0.5 ≈ 42–47 | R1@0.7 ≈ 27–32 | mAP ≈ 22–26")


## Step 6 — Training Curve

In [ ]:
import matplotlib.pyplot as plt
import re

# Parse epoch-level metrics from log
epoch_r1 = []
pattern = re.compile(r"Epoch.*?\[.*?\].*?R1@0\.5=([0-9.]+).*?mAP@0\.5=([0-9.]+)")
for line in open(LOG_PATH):
    m = pattern.search(line)
    if m:
        epoch_r1.append((float(m.group(1)), float(m.group(2))))

if epoch_r1:
    r1_vals  = [x[0] for x in epoch_r1]
    map_vals = [x[1] for x in epoch_r1]
    epochs   = list(range(1, len(r1_vals) + 1))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"M1 Training Curve — {EXP_ID}", fontsize=12)

    axes[0].plot(epochs, r1_vals, color="steelblue", lw=2)
    axes[0].axhline(53.23, color="crimson", lw=1.5, linestyle="--",
                    label="official full (53.23)")
    axes[0].set(xlabel="Epoch", ylabel="R1@0.5 (%)", title="Recall@1 IoU=0.5")
    axes[0].legend()

    axes[1].plot(epochs, map_vals, color="darkorange", lw=2)
    axes[1].axhline(30.58, color="crimson", lw=1.5, linestyle="--",
                    label="official full (30.58)")
    axes[1].set(xlabel="Epoch", ylabel="mAP@0.5 (%)", title="mAP @ IoU=0.5")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(str(RESULTS_ROOT / "m1_training_curve.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Peak R1@0.5: {max(r1_vals):.2f}%  at epoch {r1_vals.index(max(r1_vals))+1}")
else:
    print("No per-epoch metrics found in log. Training may still be running.")


## Step 7 — Saliency Sanity Check

For 5 validation samples: load the saved CLIP features, run the trained model, and visualise per-clip saliency scores against the annotated `relevant_clip_ids`. 

Expected behaviour: saliency peaks inside the annotated relevant window. If the model is working correctly, the relevant clips should have distinctly higher scores than the background clips.

**This is a critical check for the consistency pipeline** — the saliency output is what `S_saliency` in the consistency score formula depends on.

In [ ]:
import sys, torch, numpy as np, random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Add repo to path so we can import model classes
REPO_STR = str(REPO_DIR)
if REPO_STR not in sys.path:
    sys.path.insert(0, REPO_STR)

from moment_detr.inference import eval_epoch  # may not exist; use direct load below

# ── Load best checkpoint ───────────────────────────────────────────────────────
ckpt_candidates = sorted(EXP_DIR.rglob("model_best.ckpt"))
if not ckpt_candidates:
    # Fallback: use official pretrained checkpoint (for structure verification)
    ckpt_candidates = sorted(
        (PROJECT_ROOT / "moment_detr" / "run_on_video" / "moment_detr_ckpt").glob("*.ckpt")
    )

if not ckpt_candidates:
    print("No checkpoint found. Run training first (Cell 6).")
else:
    CKPT_PATH = ckpt_candidates[0]
    print(f"Loading checkpoint: {CKPT_PATH}")

    ckpt = torch.load(CKPT_PATH, map_location="cpu")
    opt  = ckpt.get("opt", {})

    print(f"  Checkpoint epoch : {ckpt.get('epoch', 'unknown')}")
    print(f"  Best R1@0.5      : {ckpt.get('metric', {}).get('MR-R1-0.5', 'unknown')}")

    # Reconstruct model from saved opt
    from moment_detr.config import BaseOptions
    from moment_detr.model import build_model

    # Build model with saved hyperparameters
    import argparse
    args = argparse.Namespace(**opt) if isinstance(opt, dict) else opt

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, criterion, contrastive_criterion = build_model(args)
    model.load_state_dict(ckpt["model"])
    model.eval().to(device)
    print(f"\nModel loaded on {device}")


In [ ]:
# ── Visualise saliency on 5 val examples ─────────────────────────────────────
val_data = load_jsonl(VAL_JSONL)
random.seed(42)
samples = random.sample(val_data, 5)

fig, axes = plt.subplots(5, 1, figsize=(14, 15))
fig.suptitle("Saliency Sanity Check — 5 Val Samples\n"
             "Blue bars = predicted saliency | Red shading = annotated relevant window",
             fontsize=11, y=1.01)

for ax, sample in zip(axes, samples):
    vid = sample["vid"]
    qid = sample["qid"]

    # Load features
    v_feat = np.load(CLIP_FEAT_DIR / f"{vid}.npz")
    v_feat = v_feat[v_feat.files[0]]                     # (n_clips, 512)
    t_feat = np.load(TEXT_FEAT_DIR / f"qid{qid}.npz")
    t_feat = t_feat["last_hidden_state"]                 # (seq_len, 512)

    # Normalise (same as dataloader)
    v_feat = v_feat / (np.linalg.norm(v_feat, axis=-1, keepdims=True) + 1e-9)
    t_feat = t_feat / (np.linalg.norm(t_feat, axis=-1, keepdims=True) + 1e-9)

    n_clips = min(len(v_feat), 75)
    v_tensor = torch.from_numpy(v_feat[:n_clips]).float().unsqueeze(0).to(device)
    t_tensor = torch.from_numpy(t_feat[:32]).float().unsqueeze(0).to(device)
    v_mask   = torch.ones(1, n_clips, dtype=torch.bool).to(device)
    t_mask   = torch.ones(1, t_tensor.shape[1], dtype=torch.bool).to(device)

    # Append TEF
    tef_st = torch.arange(n_clips).float() / n_clips
    tef_ed = tef_st + 1.0 / n_clips
    tef = torch.stack([tef_st, tef_ed], dim=1).unsqueeze(0).to(device)  # (1, n, 2)
    v_tensor = torch.cat([v_tensor, tef], dim=-1)                        # (1, n, 514)

    with torch.no_grad():
        outputs = model(
            src_vid=v_tensor, src_vid_mask=v_mask,
            src_txt=t_tensor, src_txt_mask=t_mask
        )
    saliency = outputs["saliency_scores"].squeeze(0).cpu().numpy()  # (n_clips,)

    # Normalise per-video (saliency is relative, not absolute)
    s_min, s_max = saliency.min(), saliency.max()
    saliency_norm = (saliency - s_min) / (s_max - s_min + 1e-9)

    # Plot
    clip_ids = list(range(n_clips))
    ax.bar(clip_ids, saliency_norm, color="steelblue", alpha=0.7, width=1.0)

    # Shade annotated relevant window
    for window in sample.get("relevant_windows", []):
        w_start = window[0] / 2   # seconds → clip id
        w_end   = window[1] / 2
        ax.axvspan(w_start, w_end, color="crimson", alpha=0.15)

    # Mark relevant_clip_ids
    for cid in sample.get("relevant_clip_ids", []):
        ax.axvline(cid, color="crimson", lw=0.8, alpha=0.5)

    ax.set_xlim(0, n_clips)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Norm. saliency", fontsize=8)
    q_short = sample["query"][:60] + "..." if len(sample["query"]) > 60 else sample["query"]
    ax.set_title(f'qid={qid}  |  "{q_short}"', fontsize=8)

plt.tight_layout()
plt.savefig(str(RESULTS_ROOT / "m1_saliency_sanity.png"), dpi=150, bbox_inches="tight")
plt.show()
print("\nInterpretation:")
print("  GOOD : saliency peaks align with red-shaded relevant window")
print("  BAD  : saliency is flat or peaks in non-relevant regions")
print("  → If bad, the model undertrained or the features mismatched")


## Step 8 — What These Results Mean for the Pipeline

**Interpret R1@0.5:**

| Your result | Interpretation |
|---|---|
| ≥ 47% | Strong baseline — M1 is reliable for consistency scoring |
| 42–47% | Acceptable — consistent with subset size; pipeline is valid |
| < 42% | Investigate — likely feature mismatch or training instability |

**Interpret saliency sanity check:**
- If saliency peaks align with the annotated window → `S_saliency` in the consistency score is meaningful  
- If saliency is flat → increase `--lw_saliency` (default=1.0) or check that saliency labels loaded correctly

**Next steps after this notebook:**
1. Record the best R1@0.5 and mAP numbers — these are your M1 baseline numbers for the report
2. Run M2 inference on the same val thumbnails to generate queries
3. Use `run_on_video/` with your M1 checkpoint + M2-generated queries to get pipeline outputs
4. Compute consistency score on the val set and evaluate against the proxy labels (`best_timestamp ∈ relevant_windows`)
